# Sandpile on Free Associations Network

Experiment: compare **Random** vs **Targeted** grain addition on a Free Association network using the Abelian Sandpile with stochastic dissipation (f=10⁻⁴).

**Measures:**
1. Avalanche Frequency — fraction of iterations producing at least one toppling
2. Avalanche Size — total grains moved per avalanche (Σ deg of toppled nodes)
3. Unique Toppled Nodes — number of distinct nodes involved per avalanche
4. Total Topplings — toppling events per avalanche (with repetitions)
5. Target Involvement — how often the target node is involved in avalanches

In [5]:
import sys
sys.path.insert(0, '..')

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm import tqdm

from SpreadPy.Models.sandpile import SandpileSpreading

np.random.seed(42)

## 1. Load Free Associations Network

In [6]:
# Build graph from edge list
g = nx.Graph()
with open('../toy_data/mental_lexicon_2/FreeAssociations.txt') as f:
    for line in f:
        parts = line.strip().split('	')
        if len(parts) == 2:
            g.add_edge(parts[0], parts[1])

# Remove self-loops
g.remove_edges_from(nx.selfloop_edges(g))

# Remove degree-0 nodes
isolates = list(nx.isolates(g))
g.remove_nodes_from(isolates)
print(f"Removed {len(isolates)} isolated nodes")

# Take largest connected component
largest_cc = max(nx.connected_components(g), key=len)
g = g.subgraph(largest_cc).copy()

print(f"Nodes: {g.number_of_nodes()}")
print(f"Edges: {g.number_of_edges()}")
print(f"Avg degree: {2 * g.number_of_edges() / g.number_of_nodes():.1f}")

Removed 0 isolated nodes
Nodes: 11512
Edges: 36375
Avg degree: 6.3


## 2. Load Dataset & Sample 100 Pairs

In [8]:
df = pd.read_excel('../data/all naming subjects.xlsx')

# Keep only related pairs (primecond=1), lowercase, deduplicate
related = df[df['primecond'] == 1.0][['prime', 'target']].drop_duplicates()
related['prime'] = related['prime'].str.lower()
related['target'] = related['target'].str.lower()

# Keep only pairs where both words exist in the network
nodes = set(g.nodes())
related = related[(related['prime'].isin(nodes)) & (related['target'].isin(nodes))]
print(f"Related pairs with both words in network: {len(related)}")

# Sample N pairs
pairs = related.sample(n=2, random_state=42).reset_index(drop=True)
print(f"Sampled {len(pairs)} pairs")
pairs.head(10)

Related pairs with both words in network: 1640
Sampled 2 pairs


,prime,target
0,world,earth
1,pancakes,syrup


## 3. Run Experiments

For each pair, run two sandpile simulations with N iterations:
- **Random**: add grain to a random node each step
- **Targeted**: always add grain to the cue (prime) node

In [9]:
N_ITERATIONS = 150000  # iterations per simulation
F = 1e-4             # dissipation probability

def run_sandpile(graph, n_iter, target_node=None, cue_node=None):
    """
    Run a sandpile simulation and collect avalanche metrics.
    
    :param graph: networkx graph
    :param n_iter: number of iterations
    :param target_node: if set, track whether this node is involved in avalanches
    :param cue_node: if set, always add grain to this node (targeted mode)
    :return: dict with collected metrics
    """
    model = SandpileSpreading(graph, f=F)
    
    avalanche_sizes = []
    avalanche_topplings = []
    unique_toppled = []
    target_involved = []  # True/False per iteration
    
    # iteration 0 (initial config)
    model.iteration()
    
    for _ in range(n_iter):
        result = model.iteration(node=cue_node)
        
        avalanche_sizes.append(result['avalanche_size'])
        avalanche_topplings.append(result['total_topplings'])
        unique_toppled.append(result['unique_toppled_nodes'])
        
        if target_node is not None:
            target_involved.append(target_node in result['toppled_nodes'])
    
    return {
        'avalanche_sizes': avalanche_sizes,
        'avalanche_topplings': avalanche_topplings,
        'unique_toppled': unique_toppled,
        'target_involved': target_involved,
    }

In [ ]:
results_random = []
results_targeted = []

for idx, row in tqdm(pairs.iterrows(), total=len(pairs), desc="Running experiments"):
    cue = row['prime']
    target = row['target']
    
    # Case 1: Random addition
    res_r = run_sandpile(g, N_ITERATIONS, target_node=target, cue_node=None)
    results_random.append(res_r)
    
    # Case 2: Targeted addition (always add to cue)
    res_t = run_sandpile(g, N_ITERATIONS, target_node=target, cue_node=cue)
    results_targeted.append(res_t)

Running experiments:   0%|          | 0/2 [00:00<?, ?it/s]

## 4. Analysis

### 4.1 Avalanche Frequency

Fraction of iterations that produced at least one toppling event.

In [ ]:
def avalanche_frequency(results):
    """Fraction of iterations with at least one toppling."""
    freqs = []
    for r in results:
        n_avalanches = sum(1 for t in r['avalanche_topplings'] if t > 0)
        freqs.append(n_avalanches / len(r['avalanche_topplings']))
    return freqs

freq_random = avalanche_frequency(results_random)
freq_targeted = avalanche_frequency(results_targeted)

print(f"Avalanche frequency (mean ± std):")
print(f"  Random:   {np.mean(freq_random):.4f} ± {np.std(freq_random):.4f}")
print(f"  Targeted: {np.mean(freq_targeted):.4f} ± {np.std(freq_targeted):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([freq_random, freq_targeted], labels=['Random', 'Targeted'])
ax.set_ylabel('Avalanche Frequency')
ax.set_title('Avalanche Frequency: Random vs Targeted')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Avalanche Size Distribution

In [ ]:
# Collect all non-zero avalanche sizes
sizes_random = [s for r in results_random for s in r['avalanche_sizes'] if s > 0]
sizes_targeted = [s for r in results_targeted for s in r['avalanche_sizes'] if s > 0]

def plot_powerlaw(ax, data, color, title):
    bins = np.logspace(np.log10(min(data)), np.log10(max(data)), 50)
    counts, edges = np.histogram(data, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    mask = counts > 0
    ax.scatter(centers[mask], counts[mask], s=15, color=color, alpha=0.7)
    # linear fit in log-log
    log_x = np.log10(centers[mask])
    log_y = np.log10(counts[mask])
    slope, intercept = np.polyfit(log_x, log_y, 1)
    fit_x = np.logspace(log_x.min(), log_x.max(), 100)
    ax.plot(fit_x, 10**intercept * fit_x**slope, 'k--', linewidth=1.5,
            label=f'slope = {slope:.2f}')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_powerlaw(axes[0], sizes_random, 'steelblue', 'Random Addition')
plot_powerlaw(axes[1], sizes_targeted, 'coral', 'Targeted Addition')
axes[0].set_xlabel('Avalanche Size (grains moved)')
axes[0].set_ylabel('Count')
axes[1].set_xlabel('Avalanche Size (grains moved)')
axes[1].set_ylabel('Count')
plt.suptitle('Avalanche Size Distribution', fontsize=14)
plt.tight_layout()
plt.show()

### 4.3 Total Topplings per Grain Addition

In [ ]:
topplings_random = [t for r in results_random for t in r['avalanche_topplings'] if t > 0]
topplings_targeted = [t for r in results_targeted for t in r['avalanche_topplings'] if t > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_powerlaw(axes[0], topplings_random, 'steelblue', 'Random Addition')
plot_powerlaw(axes[1], topplings_targeted, 'coral', 'Targeted Addition')
axes[0].set_xlabel('Total Topplings')
axes[0].set_ylabel('Count')
axes[1].set_xlabel('Total Topplings')
axes[1].set_ylabel('Count')
plt.suptitle('Topplings per Grain Addition', fontsize=14)
plt.tight_layout()
plt.show()

### 4.4 Number of Distinct Avalanches

In [ ]:
def count_avalanches(results):
    """Total number of non-trivial avalanches across all iterations."""
    counts = []
    for r in results:
        counts.append(sum(1 for t in r['avalanche_topplings'] if t > 0))
    return counts

n_aval_random = count_avalanches(results_random)
n_aval_targeted = count_avalanches(results_targeted)

print(f"Number of distinct avalanches per simulation (mean ± std):")
print(f"  Random:   {np.mean(n_aval_random):.1f} ± {np.std(n_aval_random):.1f}")
print(f"  Targeted: {np.mean(n_aval_targeted):.1f} ± {np.std(n_aval_targeted):.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([n_aval_random, n_aval_targeted], labels=['Random', 'Targeted'])
ax.set_ylabel('Number of Avalanches')
ax.set_title('Distinct Avalanches per Simulation: Random vs Targeted')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.5 Target Node Involvement (Targeted Case Only)

How often does the target node topple during avalanches?
Compare with how often non-target nodes topple on average.

In [ ]:
# For each pair: fraction of avalanches that involved the target
target_freq_targeted = []
for r in results_targeted:
    n_avalanches = sum(1 for t in r['avalanche_topplings'] if t > 0)
    if n_avalanches > 0:
        target_hits = sum(1 for inv, t in zip(r['target_involved'], r['avalanche_topplings']) if inv and t > 0)
        target_freq_targeted.append(target_hits / n_avalanches)
    else:
        target_freq_targeted.append(0.0)

# Same for random case (as baseline)
target_freq_random = []
for r in results_random:
    n_avalanches = sum(1 for t in r['avalanche_topplings'] if t > 0)
    if n_avalanches > 0:
        target_hits = sum(1 for inv, t in zip(r['target_involved'], r['avalanche_topplings']) if inv and t > 0)
        target_freq_random.append(target_hits / n_avalanches)
    else:
        target_freq_random.append(0.0)

print(f"Fraction of avalanches involving the target node (mean ± std):")
print(f"  Random:   {np.mean(target_freq_random):.4f} ± {np.std(target_freq_random):.4f}")
print(f"  Targeted: {np.mean(target_freq_targeted):.4f} ± {np.std(target_freq_targeted):.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([target_freq_random, target_freq_targeted], labels=['Random', 'Targeted'])
ax.set_ylabel('Fraction of avalanches involving target')
ax.set_title('Target Node Involvement in Avalanches')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.6 Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Avalanche Frequency',
        'Distinct Avalanches (per sim)',
        'Target Involvement Fraction',
    ],
    'Random (mean ± std)': [
        f"{np.mean(freq_random):.4f} ± {np.std(freq_random):.4f}",
        f"{np.mean(n_aval_random):.1f} ± {np.std(n_aval_random):.1f}",
        f"{np.mean(target_freq_random):.4f} ± {np.std(target_freq_random):.4f}",
    ],
    'Targeted (mean ± std)': [
        f"{np.mean(freq_targeted):.4f} ± {np.std(freq_targeted):.4f}",
        f"{np.mean(n_aval_targeted):.1f} ± {np.std(n_aval_targeted):.1f}",
        f"{np.mean(target_freq_targeted):.4f} ± {np.std(target_freq_targeted):.4f}",
    ],
})
summary